In [ ]:
# =============================================================================
# Exercise prob020904 – Solution of the State Equation with Sinusoidal Input
# Pure time-domain approach + symbolic integration for convolution
# No Laplace transform used
# Suitable for Springer-style supplementary material
# =============================================================================

import numpy as np
import sympy as sp

# ───────────────────────────────────────────────
# Given system parameters
# ───────────────────────────────────────────────

A = np.array([[-5, -6],
              [ 1,  0]], dtype=float)

B = np.array([[1],
              [0]], dtype=float)

q0 = np.array([[5],
               [4]], dtype=float)          # column vector (2,1)

print("System matrix A:")
print(A)
print("\nInput vector B:")
print(B)
print("\nInitial state q(0):")
print(q0)

print("\n" + "="*70)
print("The state equation is:  q̇(t) = A q(t) + B x(t)")
print("With x(t) = sin(100 t), the solution is:")
print("q(t) = exp(A t) q(0) + ∫₀ᵗ exp(A(t-τ)) B sin(100 τ) dτ\n")

# ───────────────────────────────────────────────
# Characteristic polynomial & eigenvalues (computed dynamically)
# ───────────────────────────────────────────────

char_poly_coeffs = np.poly(A)
print("Characteristic polynomial coefficients (highest degree first):")
print("   ", char_poly_coeffs)

poly_terms = []
for i, coeff in enumerate(char_poly_coeffs):
    deg = len(char_poly_coeffs) - 1 - i
    if coeff == 0:
        continue
    sign = " + " if coeff > 0 and i > 0 else " - " if coeff < 0 and i > 0 else ""
    abs_coeff = abs(coeff)
    coeff_str = f"{abs_coeff:g}" if abs_coeff != 1 or deg == 0 else ""
    if deg == 0:
        poly_terms.append(f"{sign}{coeff_str}")
    elif deg == 1:
        poly_terms.append(f"{sign}{coeff_str}λ")
    else:
        poly_terms.append(f"{sign}{coeff_str}λ^{deg}")

poly_str = "".join(poly_terms).lstrip(" +").lstrip(" -")
print(f"Characteristic equation: det(A - λI) = {poly_str} = 0")

eigenvalues = np.roots(char_poly_coeffs)
eigenvalues = np.sort(eigenvalues)[::-1]  # descending

print("\nEigenvalues:")
for i, lam in enumerate(eigenvalues, 1):
    print(f"  λ_{i} = {lam:.4g}")

print()

# ───────────────────────────────────────────────
# Method 1: β coefficients (Cayley–Hamilton approach)
# ───────────────────────────────────────────────

print("Method 1 – Cayley–Hamilton / β-coefficients approach\n")

V = np.array([[1, eigenvalues[0]],
              [1, eigenvalues[1]]], dtype=float)

V_inv = np.linalg.inv(V)

print("Vandermonde matrix V:")
print(V)
print("\nInverse V⁻¹:")
print(V_inv)

lam1 = eigenvalues[0]  # -2
lam2 = eigenvalues[1]  # -3

print("\nTherefore (with actual eigenvalues λ₁ = {:.0f}, λ₂ = {:.0f}):".format(lam1, lam2))
print(f"  β₀(t) = {V_inv[0,0]:+.4g} e^{{{lam1:.0f}t}}  {V_inv[0,1]:+.4g} e^{{{lam2:.0f}t}}")
print(f"  β₁(t) = {V_inv[1,0]:+.4g} e^{{{lam1:.0f}t}}  {V_inv[1,1]:+.4g} e^{{{lam2:.0f}t}}")

def beta(t):
    exp_lam_t = np.exp(eigenvalues * t)
    return V_inv @ exp_lam_t

# ───────────────────────────────────────────────
# Homogeneous part: exp(At) q(0) – fully computed coefficients
# ───────────────────────────────────────────────

print("\n" + "="*70)
print("Homogeneous solution: exp(At) q(0) – computed coefficients\n")

v0 = q0.flatten()          # coeff of β₀(t)
v1 = (A @ q0).flatten()    # coeff of β₁(t)

print(f"  coeff of β₀(t): {v0}")
print(f"  coeff of β₁(t): {v1}\n")

c01, c02 = V_inv[0]
c11, c12 = V_inv[1]

q1_hom_exp1 = v0[0] * c01 + v1[0] * c11
q1_hom_exp2 = v0[0] * c02 + v1[0] * c12

q2_hom_exp1 = v0[1] * c01 + v1[1] * c11
q2_hom_exp2 = v0[1] * c02 + v1[1] * c12

print("Computed symbolic form:")
print(f"  q₁(t) = {q1_hom_exp1:+.0f} e^{{{lam1:.0f}t}}  {q1_hom_exp2:+.0f} e^{{{lam2:.0f}t}}")
print(f"  q₂(t) = {q2_hom_exp1:+.0f} e^{{{lam1:.0f}t}}  {q2_hom_exp2:+.0f} e^{{{lam2:.0f}t}}")

print(f"\nWith λ₁ = {lam1:.0f}, λ₂ = {lam2:.0f}:")
print(f"  q₁(t) = {q1_hom_exp1:+.0f} e^{{-2t}}  {q1_hom_exp2:+.0f} e^{{-3t}}")
print(f"  q₂(t) = {q2_hom_exp1:+.0f} e^{{-2t}}  {q2_hom_exp2:+.0f} e^{{-3t}}")

# ───────────────────────────────────────────────
# Particular solution – e^{A(t-τ)} B – computed coefficients
# ───────────────────────────────────────────────

print("\n" + "="*70)
print("e^{A(t-τ)} B – computed coefficients\n")

b0 = B.flatten()
b1 = (A @ B).flatten()

print(f"  coeff of β₀(t-τ): {b0}")
print(f"  coeff of β₁(t-τ): {b1}\n")

q1_forced_exp1 = b0[0] * c01 + b1[0] * c11
q1_forced_exp2 = b0[0] * c02 + b1[0] * c12

q2_forced_exp1 = b0[1] * c01 + b1[1] * c11
q2_forced_exp2 = b0[1] * c02 + b1[1] * c12

print("Computed symbolic form:")
print(f"  [ {q1_forced_exp1:+.0f} e^{{{lam1:.0f}(t-τ)}}  {q1_forced_exp2:+.0f} e^{{{lam2:.0f}(t-τ)}} ]")
print(f"  [ {q2_forced_exp1:+.0f} e^{{{lam1:.0f}(t-τ)}}  {q2_forced_exp2:+.0f} e^{{{lam2:.0f}(t-τ)}} ]")

# ───────────────────────────────────────────────
# Convolution integrals I₁(t) and I₂(t) – symbolic computation with sympy
# ───────────────────────────────────────────────

print("\n" + "="*70)
print("Symbolic computation of convolution integrals I₁(t) and I₂(t)\n")

# Συντελεστές από e^{A(t-τ)} B
coeffs_q1 = [q1_forced_exp1, q1_forced_exp2]  # π.χ. -2, +3
coeffs_q2 = [q2_forced_exp1, q2_forced_exp2]  # π.χ. +1, -1

t_sym, tau = sp.symbols('t tau', real=True)
omega = 100

I1 = 0
I2 = 0

for i, alpha in enumerate([lam1, lam2]):
    coeff1 = coeffs_q1[i]
    coeff2 = coeffs_q2[i]

    gamma = -alpha
    integrand = sp.exp(gamma * tau) * sp.sin(omega * tau)
    int_def = sp.integrate(integrand, (tau, 0, t_sym))

    term1 = coeff1 * sp.exp(alpha * t_sym) * int_def
    term2 = coeff2 * sp.exp(alpha * t_sym) * int_def

    I1 += term1
    I2 += term2

I1 = sp.simplify(I1)
I2 = sp.simplify(I2)

print("I₁(t) =")
sp.pprint(I1)
print("\nI₂(t) =")
sp.pprint(I2)

print("\nThe convolution integral is:")
print("  [ I₁(t) ]")
print("  [ I₂(t) ]")

print("\nFinal state vector:")
print("q(t) = exp(At) q(0) + [I₁(t), I₂(t)]^T")

print("\nEnd of solution.")